In [3]:
from transformers import T5EncoderModel,T5Tokenizer
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_path = "Rostlab.prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_path, do_lower_case=False)
prot_t5 = T5EncoderModel.from_pretrained(model_path).to(device).eval()
for param in prot_t5.parameters():
    param.requires_grad = False

In [4]:
def transform_token2index(sequences, max_len=1200, add_special_tokens=True):
    input_ids_list = []
    attention_mask_list = []
    actual_lengths = []

    for seq in sequences:
        seq_spaced = " ".join(list(seq))
        encoding = tokenizer.encode_plus(
            seq_spaced,
            add_special_tokens=add_special_tokens,
            max_length=max_len,
            truncation=True,
            padding="max_length",
            return_attention_mask=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attn_mask = encoding['attention_mask'].squeeze(0)
        valid_len = int(attn_mask.sum().item())

        input_ids_list.append(input_ids)
        attention_mask_list.append(attn_mask)
        actual_lengths.append(valid_len)

    input_ids = torch.stack(input_ids_list).to(device)
    attention_mask = torch.stack(attention_mask_list).to(device)
    actual_lengths = torch.tensor(actual_lengths, device=device)
    return input_ids, attention_mask, actual_lengths

In [5]:
def get_esm2_embeddings(sequences, batch_size, max_len=1500,
                        pooling_strategy='mean', add_special_tokens=True,
                        return_residue=True):
    all_protein_embs = []
    all_residue_embs = []
    all_valid_lengths = []
    hidden_dim = prot_t5.config.d_model

    for i in tqdm(range(0, len(sequences), batch_size), desc="Embedding Generation"):
        batch_seqs = sequences[i:i + batch_size]
        input_ids, attention_mask, actual_lengths = transform_token2index(
            batch_seqs,
            max_len=max_len,
            add_special_tokens=add_special_tokens
        )
        with torch.no_grad():
            outputs = prot_t5(input_ids=input_ids, attention_mask=attention_mask)
            embeddings = outputs.last_hidden_state

        for j in range(len(batch_seqs)):
            actual_len = actual_lengths[j].item()
            all_valid_lengths.append(actual_len)

            residue_emb = embeddings[j, :actual_len, :]

            if return_residue:
                padded_residue = torch.zeros(max_len, hidden_dim, device='cpu')
                padded_residue[:actual_len] = residue_emb.cpu()
                all_residue_embs.append(padded_residue.numpy())

            if pooling_strategy == 'mean':
                protein_emb = residue_emb.mean(dim=0)
            elif pooling_strategy == 'max':
                protein_emb = residue_emb.max(dim=0)[0]
            else:
                raise ValueError(f"Pooling is not supported {pooling_strategy}")

            all_protein_embs.append(protein_emb.cpu().numpy())

    result = {}
    result['protein_embeddings'] = np.array(all_protein_embs)
    result['valid_lengths'] = np.array(all_valid_lengths)
    if return_residue:
        result['residue_embeddings'] = np.array(all_residue_embs)
    return result

In [7]:
def save_to_csv(data_dict, output_path):
    protein_embs = data_dict['protein_embeddings']
    df = pd.DataFrame(protein_embs)
    df.columns = [f'emb_{i}' for i in range(protein_embs.shape[1])]
    df.to_csv(output_path, index=False)
    print(f"\n{'=' * 60}")
    print(f"data has been saved to: {output_path}")
    print(f"protein_embeddings: {protein_embs.shape}")
    print(f"{'=' * 60}")

In [6]:
def load_csv(file_path):
    df = pd.read_csv(file_path)
    print(f"\nLoading file: {file_path}, shape={df.shape}")
    return df

def print_shape_info(data_dict):
    print(f"\n{'=' * 60}")
    print("output shape:")
    print(f"protein_embeddings: {data_dict['protein_embeddings'].shape}")
    print(f"{'=' * 60}")

In [8]:
def main():
    csv_path = 'demo/example_data/example_data.csv'
    df = pd.read_csv(csv_path)
    protein_series = df['protein']
    sequences = protein_series.tolist()

    print(f"Total number of protein sequences in the dataset: {len(sequences)}")
    print(f"Article 1: Example Sequence: {sequences[0][:30]}...")
    result = get_esm2_embeddings(
        sequences,
        max_len=1200,
        batch_size=8,
        pooling_strategy='mean',
        add_special_tokens=True,
        return_residue=False
    )
    print_shape_info(result)
    save_to_csv(
        data_dict=result,
        output_path='example_data_protT5_out.csv'
    )
    return result
loaded_data = load_csv('example_data_protT5_out.csv')
print(f"protein_emb shape: {loaded_data.values[:, 1:].shape}")


Loading file: example_data_protT5_out.csv, shape=(9, 1025)
protein_emb shape: (9, 1024)
